# WEEX V6.54 — MARKET/LIMIT i shadow tracking
Wspolna historia OI i wynikow, pobranie ZIP; zadnych automatycznych zlecen.

In [ ]:
import requests
import shutil
from pathlib import Path
from google.colab import drive, files

# Trwala historia OI i obserwowanych wynikow
drive.mount("/content/drive")
oi_dir = Path("/content/drive/MyDrive/WEEX")
oi_dir.mkdir(parents=True, exist_ok=True)
state = oi_dir / "oi_history_shared.json"
if not state.exists():
    old = [oi_dir / f"weex_v{v}_oi_history_state.json" for v in
           (653,652,651,650,649,648,647,646,645,644)]
    found = [p for p in old if p.exists()]
    if found:
        shutil.copy2(max(found, key=lambda p:p.stat().st_mtime), state)

# Pobranie kolektora z GitHub
url = "https://raw.githubusercontent.com/danieldziezak1986-hash/weex-futures/main/weex_v654_collector.py"
r = requests.get(url,timeout=30)
r.raise_for_status()
Path("weex_v654_collector.py").write_bytes(r.content)

# Test poprawnosci
%run weex_v654_collector.py --self-test

# TOP 150 Crypto + TOP 50 TradFi / MARKET + LIMIT
%run weex_v654_collector.py --top 150 --tradfi-top 50 --max-deep 30 --limit-ttl-minutes 180 --state-path /content/drive/MyDrive/WEEX/oi_history_shared.json

# Automatyczny ZIP
folder = Path("/content/weex_v654_results")
archiwa = list(folder.glob("weex_v654_*.zip"))
if not archiwa:
    raise FileNotFoundError("Nie znaleziono ZIP WEEX V6.54")
najnowszy = max(archiwa, key=lambda p:p.stat().st_mtime)
print("Pobieranie wynikow:",najnowszy.name)
files.download(str(najnowszy))
